In [1]:
print("Crop yield prediction using ML")

Crop yield prediction using ML


In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler

In [3]:
# Loading Dataset
data = pd.read_csv("/kaggle/input/datasets/vanshikajain05/crop-yield-dataset/crop_csv_file.csv")
print(data.columns)

print(data.head())

Index(['State_Name', 'District_Name', 'Crop_Year', 'Season', 'Crop',
       'Temperature', 'Humidity', 'Soil Moisture', 'Area', 'Production'],
      dtype='object')
                    State_Name District_Name  Crop_Year       Season  \
0  Andaman and Nicobar Islands      NICOBARS       2000  Kharif        
1  Andaman and Nicobar Islands      NICOBARS       2000  Kharif        
2  Andaman and Nicobar Islands      NICOBARS       2000  Kharif        
3  Andaman and Nicobar Islands      NICOBARS       2000  Whole Year    
4  Andaman and Nicobar Islands      NICOBARS       2000  Whole Year    

                  Crop  Temperature  Humidity  Soil Moisture    Area  \
0             Arecanut           36        35             45  1254.0   
1  Other Kharif pulses           37        40             46     2.0   
2                 Rice           36        41             50   102.0   
3               Banana           37        42             55   176.0   
4            Cashewnut           36       

In [4]:
# Data Cleaning

# Remove invalid rows
data = data[(data['Area'] > 0) & (data['Production'] > 0)]

# Create Yield
data['Yield'] = data['Production'] / data['Area']
# Replace Yield with its logarithmic value
data['Yield'] = np.log1p(data['Yield'])
# Remove outliers
data = data[data['Yield'] < data['Yield'].quantile(0.90)]

# Replace inf and -inf values with NaN
data.replace([np.inf, -np.inf], np.nan, inplace=True)
# Remove NaN values
data = data.dropna()

In [5]:
# Encoding
# Convert text (categorical data) into numbers using one-hot encoding
data = pd.get_dummies(data, columns=['Crop', 'State_Name', 'Season'])

# Drop district_name
data = data.drop(columns=['District_Name'], errors='ignore')

# show correlation
corr = data.corr()
print(corr['Yield'].sort_values(ascending=False))



Yield                      1.000000
Crop_Potato                0.354148
Season_Whole Year          0.295088
Crop_Mesta                 0.254198
Crop_Jute                  0.245026
                             ...   
State_Name_Chhattisgarh   -0.125543
Crop_Urad                 -0.133090
Crop_Sesamum              -0.153145
Season_Rabi               -0.172604
Crop_Moong(Green Gram)    -0.176534
Name: Yield, Length: 94, dtype: float64


In [6]:
# Dropping columns
data = data.drop(columns=['Temperature','Humidity','Soil Moisture'], errors='ignore')
X = data.drop(columns=['Yield', 'Production'])
y = data['Yield']
# Convert all input features to a common scale (normalize them
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [7]:
# Linear Regression
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
# R2 Score
score = r2_score(y_test, y_pred)
print("R2 Score:", score)

R2 Score: 0.7700699332645666


In [8]:
from sklearn.svm import SVR
from sklearn.metrics import r2_score

# Create SVM model
svm_model = SVR(kernel='rbf')

# Train model
svm_model.fit(X_train, y_train)

# Predict
y_pred_svm = svm_model.predict(X_test)

# Accuracy
svm_r2 = r2_score(y_test, y_pred_svm)
print("SVM R2 Score:", svm_r2)

SVM R2 Score: 0.8893874011630425
